## **Build Tool Calling Agent**

In [1]:
import re
from pytube import YouTube
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

d:\Projects\ai-math-assistant-with-LangChain-and-LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
bits_and_bytes_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3"
)

llm_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config = bits_and_bytes_config,
    
).to(device)

Loading weights: 100%|██████████| 291/291 [00:40<00:00,  7.23it/s]


In [5]:
pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=tokenizer
)

In [6]:
hf_pipe = HuggingFacePipeline(
    pipeline=pipe
)

llm = ChatHuggingFace(llm=hf_pipe)

### Step 01:

Video ID extraction tool

In [7]:
from langchain_core.tools import tool

In [8]:
@tool
def video_id_extractor(url:str) -> str:
    
    """extract 11 charcter video id from the youtube video url
    
    Args:
    - url (str): A youtube video url that contains the video id
    
    return
    - (str) : extracted video id or error message if failed to extracted the video id
    """
    
    pattern = r'https:\/\/\www\.youtube\.com\/watch\?v=([a-zA-Z0-9_-]{11})'
    
    match = re.search(pattern,url)
    
    return match.group(1) if match else "Error: Invalid YouTube URL"

In [9]:
print(video_id_extractor.name)
print(video_id_extractor.description)
print(video_id_extractor.args)
print(video_id_extractor.func)

video_id_extractor
extract 11 charcter video id from the youtube video url

    Args:
    - url (str): A youtube video url that contains the video id

    return
    - (str) : extracted video id or error message if failed to extracted the video id
{'url': {'title': 'Url', 'type': 'string'}}
<function video_id_extractor at 0x0000021166759DA0>


In [10]:
video_id_extractor.run("https://www.youtube.com/watch?v=hfIUstzHs9A")

'hfIUstzHs9A'

In [11]:
tools = []

In [12]:
tools.append(video_id_extractor)

### Step 02:

Transcript fetching

In [13]:
from youtube_transcript_api import YouTubeTranscriptApi

In [14]:
@tool
def transcript_fetch(video_id:str, language:str = "en") -> str:
    
    """Fetch the transcript of the youtube video
    
    Args:
        video_id (str): The YouTube video ID (e.g., "dQw4w9WgXcQ").
        language (str): Language code for the transcript (e.g., "en", "es").
    
    Returns:
        str: The transcript text or an error message.
    """
    
    yt_transcript = YouTubeTranscriptApi()
    transcript = yt_transcript.fetch(video_id,languages=[language])
    text = " ".join(snippet.text for snippet in transcript.snippets)
    
    return text

In [15]:
transcript_fetch.run("hfIUstzHs9A")

'Over the past couple of months, large language models, or LLMs, such as chatGPT, have taken the world by storm. Whether it\'s writing poetry or helping plan your upcoming vacation, we are seeing a step change in the performance of AI and its potential to drive enterprise value. My name is Kate Soule. I\'m a senior manager of business strategy at IBM Research, and today I\'m going to give a brief overview of this new field of AI that\'s emerging and how it can be used in a business setting to drive value. Now, large language models are actually a part of a different class of models called foundation models. Now, the term "foundation models" was actually first coined by a team from Stanford when they saw that the field of AI was converging to a new paradigm. Where before AI applications were being built by training, maybe a library of different AI models, where each AI model was trained on very task-specific data to perform very specific task. They predicted that we were going to start 

In [56]:
tools.append(transcript_fetch)

### Step 03:

Youtube Search Tool

In [38]:
from pytube import Search
from IPython.display import display, JSON

In [40]:
from yt_dlp import YoutubeDL

In [47]:
@tool
def youtube_search(query:str) -> list[dict[str,str]]:
    """Search youtube videos that matches to query.
    
    Args:
    - query (str): The string that use to search the yotube video
    
    Returns:
    List of dictionaries that contains video title and video IDs in format:
    [{'title': 'Video Title', 'video_id': 'abc123'}, ...]
    Return error message if search fails.
     
    """
    
    ydl_opts = {
    "quiet": True,
    "extract_flat": True
    }
    
    with YoutubeDL(ydl_opts) as ydl:
        result = ydl.extract_info(
            f"ytsearch10:{query}",
            download=False
        )
        
    videos = []
    
    for entry in result['entries']:
        videos.append(
            {
                "title": entry["title"],
                "video_id": entry["id"],
                "url": f"https://youtu.be/{entry['id']}"
            }
        )
    
    return videos

In [50]:
search_result = youtube_search.run("How to run a python environment")


In [53]:
search_result

[{'title': 'Python Virtual Environments - Full Tutorial for Beginners',
  'video_id': 'Y21OR1OPC9A',
  'url': 'https://youtu.be/Y21OR1OPC9A'},
 {'title': 'The Complete Guide to Python Virtual Environments!',
  'video_id': 'KxvKCSwlUv8',
  'url': 'https://youtu.be/KxvKCSwlUv8'},
 {'title': 'Python Virtual Environment and pip for Beginners',
  'video_id': 'eDe-z2Qy9x4',
  'url': 'https://youtu.be/eDe-z2Qy9x4'},
 {'title': 'How To Setup A Virtual Environment For Python In Visual Studio Code In 2023',
  'video_id': 'GZbeL5AcTgw',
  'url': 'https://youtu.be/GZbeL5AcTgw'},
 {'title': 'Setting Up A Python Environment for Data Analysis and Machine Learning',
  'video_id': 'NDFMa5FSQuI',
  'url': 'https://youtu.be/NDFMa5FSQuI'},
 {'title': 'Create a Python virtual environment in Windows with VS Code',
  'video_id': 'JYdd1k44FRM',
  'url': 'https://youtu.be/JYdd1k44FRM'},
 {'title': "You MUST WATCH THIS before installing PYTHON. PLEASE DON'T MAKE this MISTAKE.",
  'video_id': '28eLP22SMTA',
  'u

In [51]:
display(JSON(search_result))

<IPython.core.display.JSON object>

In [54]:
tools.append(youtube_search)

In [57]:
tools

[StructuredTool(name='video_id_extractor', description='extract 11 charcter video id from the youtube video url\n\n    Args:\n    - url (str): A youtube video url that contains the video id\n\n    return\n    - (str) : extracted video id or error message if failed to extracted the video id', args_schema=<class 'langchain_core.utils.pydantic.video_id_extractor'>, func=<function video_id_extractor at 0x0000021166759DA0>),
 StructuredTool(name='youtube_search', description="Search youtube videos that matches to query.\n\n    Args:\n    - query (str): The string that use to search the yotube video\n\n    Returns:\n    List of dictionaries that contains video title and video IDs in format:\n    [{'title': 'Video Title', 'video_id': 'abc123'}, ...]\n    Return error message if search fails.", args_schema=<class 'langchain_core.utils.pydantic.youtube_search'>, func=<function youtube_search at 0x000002124C7EC0E0>),
 StructuredTool(name='transcript_fetch', description='Fetch the transcript of t

### Step 04:
Metadata extractor

In [58]:
@tool
def meta_data_extract(url:str) -> dict:
    
    """Extract meta data of a given youtube video link, including title, views, duration, channel, likes, comments, and chapters.
    """
    
    with YoutubeDL({'quiet':True}) as ydl:
        info = ydl.extract_info(url, download=False)
        
        return{
            "title": info.get('title'),
            'views': info.get('view_count'),
            'duration': info.get('duration'),
            'channel': info.get('uploader'),
            'likes': info.get('like_count'),
            'comments': info.get('comment_count'),
            'chapters': info.get('chapters', [])
        }

In [59]:
meta_data_extract.run("https://youtu.be/qWHaMrR5WHQ")

{'title': 'Explained how to use Tools (tool calling) in LangChain',
 'views': 1514,
 'duration': 1048,
 'channel': 'Praveen Reddy Learnings',
 'likes': 25,
 'comments': 5,
 'chapters': None}

In [60]:
tools.append(meta_data_extract)

Step 04:
Get trending video 

In [64]:
@tool
def get_trending_videos(region_code: str) -> list[dict]:
    """Fetch currently trending videos in given region code and retrun as list of dictionaries or retrun error message if there have error in fetching"""

    try:
        with YoutubeDL(
            {
                "geo_bypass": True,
                "geo_bypass_country": region_code.upper(),
                "quiet": True,
                "extract_flat": True,
                "skip_download": True,
            }
        ) as ydl:
            info = ydl.extract_info(
                "https://www.youtube.com/feed/trending", download=False
            )
            entries = info.get("entries", [])
            trendings = []

            for entry in entries:
                video_data = {
                    "title": entry.get("title"),
                    "video_id": entry.get("id"),
                    "url": f"https://youtu.be/{entry.get('id')}",
                }

                trendings.append(video_data)

            return trendings
    except Exception as e:
        return f"{str(e)}"

In [66]:
get_trending_videos.run("JP")

ERROR: [youtube:tab] trending: The channel/playlist does not exist and the URL redirected to youtube.com home page


'ERROR: [youtube:tab] trending: The channel/playlist does not exist and the URL redirected to youtube.com home page'

In [67]:
tools.append(get_trending_videos)

In [69]:
@tool
def get_tumbnails(url: str) -> list[dict]:
    """Extract the tumbnail metadata from the given youtube url and return dict type output"""

    try:
        
        with YoutubeDL(
            {
                "quiet": True,
            }
        ) as ydl:
            info = ydl.extract_info(url, download=False)

            entries = info.get("thumbnail", [])

            thumbnails = []

            for t in entries:
                if url in t:
                    thumbnails.append(
                        {
                            "url": t["url"],
                            "width": t.get("width"),
                            "height": t.get("height"),
                            "resolution": f"{t.get('width', '')}x{t.get('height', '')}".strip(
                                "x"
                            ),
                        }
                    )
            
            return thumbnails
        
    except Exception as e:
        return [{"error": f"Failed to get thumbnails: {str(e)}"}]

In [75]:
#result = get_tumbnails.run("https://www.youtube.com/watch?v=0g3r-WoqE2A&list=RD4qwkctf14HU&index=2")

In [76]:
tools.append(get_tumbnails)

Binding Tools to LLM

In [77]:
llm_with_tools = llm.bind_tools(tools)

In [78]:
for tool in tools:
    schema = {
        "name": tool.name,
        "description": tool.description,
        "parameters": tool.args_schema.schema() if tool.args_schema else {},
        "return": tool.return_type if hasattr(tool,"return_type") else None
    }
    display(JSON(schema))

C:\Users\Vish\AppData\Local\Temp\ipykernel_16664\4222798368.py:5: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  "parameters": tool.args_schema.schema() if tool.args_schema else {},


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

In [79]:
query = "I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english"
print(query)

I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english


In [81]:
from langchain_core.messages import HumanMessage

In [106]:
chat_history = [HumanMessage(content=query)]
print(chat_history)

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]


In [100]:
from langchain_openai import ChatOpenAI
import os
import dotenv

In [102]:
dotenv.load_dotenv()
api_key = os.getenv("OPENAI_KEY")


In [108]:
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    api_key=api_key,
    temperature=0.7
)

In [104]:
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000021294EF4710>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021255354AD0>, root_client=<openai.OpenAI object at 0x0000021294FBC0D0>, root_async_client=<openai.AsyncOpenAI object at 0x00000212959CAF50>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None)

In [105]:
llm_with_tools = llm.bind_tools(tools)

In [109]:
response_1 = llm_with_tools.invoke(chat_history)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [84]:
response_1

AIMessage(content='<s>[INST] I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english[/INST]Title: "Quantum Computing Explained with a Cup of Coffee"\n\nThe video "Quantum Computing Explained with a Cup of Coffee" provides an easy-to-understand explanation of quantum computing, using a simple analogy of a cup of coffee and a deck of cards.\n\nThe video begins by explaining the difference between classical and quantum computing. Classical computers use bits, which are like switches that can be either on or off, represented as 0 or 1. Quantum computers, on the other hand, use quantum bits, or qubits, which can be in multiple states at once thanks to the principles of quantum mechanics, specifically superposition and entanglement.\n\nSuperposition is likened to a cup of coffee with cream that can be in multiple states (black, half-black, half-white, or white) until the coffee is stirred, at which point it becomes just black or white. Entanglement is explai

In [85]:
chat_history.append(response_1)

In [93]:
chat_history

[HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={})]

In [89]:
tool_calls_1 = response_1.tool_calls
tool_calls_1

[]

In [90]:
from langgraph.prebuilt import create_react_agent

In [95]:
agent_1 = create_react_agent(llm,tools, debug=True)

In [98]:
agent_1.invoke({
    "messages": [("human", query)]
})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[values] {'messages': [HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={}, id='53f8dea2-b32a-44f4-966a-ea923be356bf')]}
[updates] {'agent': {'messages': [AIMessage(content='<s>[INST] I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english[/INST]Title: "The Science of Happiness: Jeanne Muhammad\'s TEDx Talk"\n\nSummary:\n\nIn this TEDx Talk, Jeanne Muhammad discusses the science of happiness and offers practical strategies for individuals to cultivate and maintain happiness in their lives. She explains that while happiness is often thought to be the result of external factors such as wealth, success, or relationships, research shows that it is largely influenced by internal factors like our mindset and habits.\n\nTo support her argument, Muhammad cites various studies and scientific findings. For instance, she mentions a study that found that wealth

{'messages': [HumanMessage(content='I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english', additional_kwargs={}, response_metadata={}, id='53f8dea2-b32a-44f4-966a-ea923be356bf'),
  AIMessage(content='<s>[INST] I want to summarize youtube video: https://www.youtube.com/watch?v=T-D1OfcDW1M in english[/INST]Title: "The Science of Happiness: Jeanne Muhammad\'s TEDx Talk"\n\nSummary:\n\nIn this TEDx Talk, Jeanne Muhammad discusses the science of happiness and offers practical strategies for individuals to cultivate and maintain happiness in their lives. She explains that while happiness is often thought to be the result of external factors such as wealth, success, or relationships, research shows that it is largely influenced by internal factors like our mindset and habits.\n\nTo support her argument, Muhammad cites various studies and scientific findings. For instance, she mentions a study that found that wealth and happiness have a limited relationship

In [110]:
tools_map = {
    "get_thumbnails" : get_tumbnails,
    "get_trending_videos": get_trending_videos,
    "extract_video_id": video_id_extractor,
    "fetch_transcript": transcript_fetch,
    "search_youtube": youtube_search,
    "get_full_metadata": meta_data_extract
}

In [111]:
tools_call = response_1.tool_calls

In [113]:
# tool_name = tools_call[0]['name']
# tool_call_id = tools_call[0]['id']
# args = tools_call[0]['args']


In [ ]:
# my_tool = tools_map['tool_name']
# video_id = my_tool.invoke(args)

In [114]:
from langchain_core.messages import ToolMessage

In [ ]:
# chat_history.append(ToolMessage(content=video_id, tool_call_id=tools_call[0]['id']))